In [ ]:
import pickle
from pathlib import Path
import glob
from capymoa.drift.eval_detector import EvaluateDriftDetector

# Get the most recent dd_metrics.pkl file.
filenames = glob.glob("logs/*/*/*/*/*/dd_metrics.pkl")
filenames = sorted(filenames, key=lambda x: Path(x).stat().st_mtime)
filename = filenames[-1]
filename = "logs/evaluate/slow/EWC/ADWIN_JOINT/001/dd_metrics.pkl"

with open(filename, "rb") as f:
    metrics = pickle.load(f)
print(filename)

In [ ]:
n = len(metrics["ce_stream"])
max_delay = int(n / 5)
max_early_detection = max_delay

In [ ]:
EvaluateDriftDetector(
    max_delay=max_delay, max_early_detection=00, rate_period=n // 5
).calc_performance(
    metrics["trues"],
    metrics["preds"],
    n,
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


# Windowed mean
window_size = 516

scenario, strategy, detector = filename.split("/")[2:5]


def calculate_stream_statistics(window_size: int, data: np.ndarray):
    # Create the kernel for the moving average
    kernel = np.ones(window_size) / window_size

    # Calculate moving mean (valid mode returns only full-window overlaps)
    mean = np.convolve(data, kernel, mode="valid")

    # Calculate moving mean of squares
    mean_sq = np.convolve(data**2, kernel, mode="valid")

    # Calculate moving standard deviation
    # np.maximum is used to prevent negative values due to floating point errors
    std = np.sqrt(np.maximum(mean_sq - mean**2, 0))

    # Define x-axis based on the valid convolution output length
    x = np.arange(len(mean))

    return mean, std, x


ce_stream = metrics["ce_stream"]
ce_stream_avg, ce_stream_std, ce_stream_x = calculate_stream_statistics(
    window_size, ce_stream
)
error_stream = metrics["error_stream"]
error_stream_avg, error_stream_std, error_stream_x = calculate_stream_statistics(
    window_size, error_stream
)

# Downsample the x-axis for better visualization if needed
step = 200
error_stream_x = error_stream_x[::step]
error_stream_avg = error_stream_avg[::step]
error_stream_std = error_stream_std[::step]
ce_stream_x = ce_stream_x[::step]
ce_stream_avg = ce_stream_avg[::step]
ce_stream_std = ce_stream_std[::step]

# x = np.linspace(0, len(ce_stream), len(error_stream_avg))

# Plot both the Error stream and the CE loss stream with different y-axes.
fig, ax1 = plt.subplots()
ax1.set_title(f"{scenario} - {strategy} - {detector}")

ax1.set_xlabel("Instance")
ax1.set_ylabel("Error (Windowed)", color="blue")
ax1.plot(error_stream_x, error_stream_avg, color="blue")
# ax1.fill_between(
#     error_stream_x,
#     np.array(error_stream_avg) - np.array(error_stream_std),
#     np.array(error_stream_avg) + np.array(error_stream_std),
#     color="blue",
#     alpha=0.1,
# )
ax1.tick_params(axis="y", labelcolor="blue")

ax2 = ax1.twinx()
ax2.set_ylabel("Cross-Entropy Loss (Windowed)", color="orange")
ax2.plot(ce_stream_x, ce_stream_avg, color="orange")
# ax2.fill_between(
#     ce_stream_x,
#     np.array(ce_stream_avg) - np.array(ce_stream_std),
#     np.array(ce_stream_avg) + np.array(ce_stream_std),
#     color="orange",
#     alpha=0.1,
# )
ax2.tick_params(axis="y", labelcolor="orange")

# set ranges
ax1.set_ylim(0, 1)
ax2.set_ylim(2.5, 3)

ymin, ymax = plt.ylim()


# plt.vlines(metrics["preds"], ymin, ymax, colors="green", alpha=0.5)

plt.vlines(metrics["trues"], ymin, ymax, colors="red", label="true drift")
plt.vlines(
    np.array(metrics["trues"]) - metrics["max_delay"] / 2, ymin, ymax, colors="orange"
)
plt.vlines(
    np.array(metrics["trues"]) + metrics["max_delay"] / 2, ymin, ymax, colors="orange"
)

In [ ]:
max_delay

In [ ]:
with (Path(filename).parent / "ocl_metrics.pkl").open("rb") as f:
    ocl_metrics = pickle.load(f)

vmax = 1
assert ocl_metrics.accuracy_matrix.max() < vmax
plt.imshow(ocl_metrics.accuracy_matrix, aspect="equal", cmap="grey", vmin=0, vmax=vmax)
plt.ylabel("Train Task")
plt.xlabel("Test Task")

n = 5
# # Add arrow annotations for forward transfer, backward transfer, and retention
# plt.text(0.5*n, 0.2*n, "Forward Transfer", ha="center", va="center", color="white")
# plt.text(0.2*n, 0.5*n, "Backward Transfer", ha="center", va="center", color="white")